# Operaciones de bits

Un entero se puede ver como un vector de bits y operar con todos a la vez en $O(1)$ con `&` (y), `|` (o), `^` (o exclusivo), `~` (negación) y los desplazamientos `<<` y `>>`. Algunas identidades útiles:

$$x \mathbin{\&} (x - 1) \text{ borra el 1 más bajo}, \qquad x \mathbin{\&} (-x) \text{ lo aísla}, \qquad a \oplus a = 0, \qquad a \oplus 0 = a, \qquad \sim x = -x - 1$$

En Python los enteros no tienen tamaño fijo: los negativos se comportan como si tuvieran infinitos unos a la izquierda (complemento a dos), y por eso `~x` es negativo.

## 1. Operadores bit a bit

Mostrar en binario el resultado de `&`, `|`, `^`, `~`, `<<` y `>>` para dos números, y comprobar que desplazar $k$ bits equivale a multiplicar o dividir (por abajo) por $2^k$ y que `~x == -x - 1`.

In [1]:
a, b = 0b1100, 0b1010
operaciones = {"a": a, "b": b, "a & b": a & b, "a | b": a | b, "a ^ b": a ^ b, "a << 2": a << 2, "a >> 2": a >> 2}
for nombre, valor in operaciones.items():
    print(f"{nombre:7} {valor:08b} = {valor}")
print(f"~a = {~a}, que en 8 bits es {~a & 0xFF:08b}")

for x in range(-100, 100):
    assert ~x == -x - 1
    for k in range(6):
        assert x << k == x * 2 ** k and x >> k == x // 2 ** k

a       00001100 = 12
b       00001010 = 10
a & b   00001000 = 8
a | b   00001110 = 14
a ^ b   00000110 = 6
a << 2  00110000 = 48
a >> 2  00000011 = 3
~a = -13, que en 8 bits es 11110011


## 2. Contar bits a 1

Contar los bits a 1 de un entero no negativo recorriéndolos con `>>`, y con el truco de Kernighan: `x & (x - 1)` borra el 1 más bajo, así que el bucle da tantas vueltas como unos haya. Comprobar que coinciden con `int.bit_count()` y `bin(x).count("1")`, y usarlo para calcular la distancia de Hamming entre dos números (bits en los que se diferencian).

In [2]:
def bits_desplazando(x):
    total = 0
    while x:
        total += x & 1
        x >>= 1
    return total

def bits_kernighan(x):
    total = 0
    while x:
        x &= x - 1
        total += 1
    return total

for x in range(5000):
    assert bits_desplazando(x) == bits_kernighan(x) == x.bit_count() == bin(x).count("1")

def hamming(a, b):
    return bits_kernighan(a ^ b)

x = 0b1011_0100
print(f"{x:b} tiene {bits_kernighan(x)} unos")
print(f"hamming({0b1011:04b}, {0b0110:04b}) = {hamming(0b1011, 0b0110)}")

10110100 tiene 4 unos
hamming(1011, 0110) = 3


## 3. Potencia de 2

Una potencia de 2 tiene un único bit a 1, así que `x & (x - 1)` la deja a 0. Escribir `es_potencia_de_2` y la menor potencia de 2 mayor o igual que `x` con `int.bit_length()`, y comprobarlas contra el conjunto de potencias.

In [3]:
def es_potencia_de_2(x):
    return x > 0 and x & (x - 1) == 0

def siguiente_potencia_de_2(x):
    return 1 << (x - 1).bit_length() if x > 1 else 1

potencias = {2 ** k for k in range(20)}
for x in range(-10, 2 ** 19 + 1):
    assert es_potencia_de_2(x) == (x in potencias)
for x in range(1, 5000):
    assert siguiente_potencia_de_2(x) == min(p for p in potencias if p >= x)
print([x for x in range(70) if es_potencia_de_2(x)], siguiente_potencia_de_2(100))

[1, 2, 4, 8, 16, 32, 64] 128


## 4. Elemento único con XOR

En una lista donde todos los números aparecen dos veces salvo uno, encontrarlo en $O(n)$ y $O(1)$ memoria: al combinarlo todo con XOR las parejas se anulan. Comprobarlo con `Counter`.

In [4]:
import random
from collections import Counter
from functools import reduce
from operator import xor

def unico(lista):
    return reduce(xor, lista)

random.seed(4)
for _ in range(1000):
    valores = random.sample(range(-50, 50), random.randint(1, 10))
    lista = valores[1:] * 2 + valores[:1]
    random.shuffle(lista)
    assert unico(lista) == next(x for x, c in Counter(lista).items() if c == 1) == valores[0]
print(unico([4, 1, 2, 1, 2]))

4


## 5. Subconjuntos con máscaras

Cada entero $m$ de $0$ a $2^n - 1$ representa un subconjunto: el elemento $i$ está si el bit $i$ de $m$ vale 1. Generar así todos los subconjuntos, compararlos con `itertools.combinations` y usarlos para encontrar los subconjuntos que suman un objetivo.

In [5]:
from itertools import combinations

def subconjuntos(elementos):
    n = len(elementos)
    return [[elementos[i] for i in range(n) if m >> i & 1] for m in range(1 << n)]

for m, s in enumerate(subconjuntos(["a", "b", "c"])):
    print(f"{m:03b} {s}")

for n in range(9):
    elementos = list(range(n))
    esperado = sorted(list(c) for k in range(n + 1) for c in combinations(elementos, k))
    assert sorted(subconjuntos(elementos)) == esperado

numeros = [3, 34, 4, 12, 5, 2]
print([s for s in subconjuntos(numeros) if sum(s) == 9])

000 []
001 ['a']
010 ['b']
011 ['a', 'b']
100 ['c']
101 ['a', 'c']
110 ['b', 'c']
111 ['a', 'b', 'c']
[[4, 5], [3, 4, 2]]


## 6. Intercambio sin variable auxiliar

Intercambiar dos enteros sin variable auxiliar con tres XOR: $a \leftarrow a \oplus b$, $b \leftarrow a \oplus b$ (que vale el $a$ original) y $a \leftarrow a \oplus b$ (que vale el $b$ original). Comprobarlo y ver por qué falla si las dos variables son la misma posición de una lista. En Python lo natural es `a, b = b, a`.

In [6]:
def intercambia_xor(lista, i, j):
    lista[i] ^= lista[j]
    lista[j] ^= lista[i]
    lista[i] ^= lista[j]

for _ in range(1000):
    a, b = random.randint(-1000, 1000), random.randint(-1000, 1000)
    lista = [a, b]
    intercambia_xor(lista, 0, 1)
    assert lista == [b, a]

lista = [7, 9]
intercambia_xor(lista, 0, 1)
print(lista)
lista = [7, 9]
intercambia_xor(lista, 0, 0)
print(lista, "← con i == j el primer XOR deja el valor a 0")

[9, 7]
[0, 9] ← con i == j el primer XOR deja el valor a 0
